# TartanAir 640 — đóng gói lại shard 1 và 2 từ Output đã lưu (không tải lại)

**Lỗi.** Output đã lưu của hai lần dựng **đủ file**: shard 1 (`notebookf5b145bacc`, phiên bản 1) có
46.564 file, shard 2 (phiên bản 2) có 46.941 file, khớp từng quỹ đạo với `shard1.json` / `shard2.json`.
Hai lần đó chạy builder cũ (`93a2cc9`), chưa đóng gói, nên Output là ~47 nghìn file lẻ. Nút
**New Dataset** từ Output chỉ nhận **500 file đầu** theo thứ tự đường dẫn: 3 file `build_meta`
(729,45 KB) + 497 file đầu của `tartanair640/` (~172 MB). Phần còn lại không bao giờ vào dataset.

**Cách sửa.** Notebook này không tải gì từ Hugging Face. Nó:
1. gắn **đúng phiên bản** Output cũ làm input (`kagglehub.notebook_output_download`), chỉ đọc, không sửa;
2. kiểm tra từng quỹ đạo với báo cáo lúc dựng: số khung, tổng byte ảnh, `frames.npy`, IMU, `build_done.json`;
3. ghi mỗi môi trường thành **một file `.tar`** (không nén: WebP đã nén), băm MD5 từng file khi ghi,
   **đọc lại toàn bộ tar** và so MD5 trước khi đổi tên thành `.tar`;
4. ghi `build_meta/inventory_shard<N>.tsv` (đường dẫn, byte, MD5 của mọi file) để kiểm dataset sau này.

Output còn ~27 file: `tartanair640/<môi trường>.tar` + `build_meta/`. Khi tạo dataset, Kaggle giải nén mỗi
tar thành thư mục cùng tên (đã thấy với 24 tar của shard 0: dataset có đủ 46.700 file, khớp từng byte),
nên dataset đọc thành `tartanair640/<môi trường>/Data_easy/P000/...`.

## Chạy
1. Kaggle → *New Notebook* → *File → Import Notebook* → chọn file này. *Settings*: **Accelerator: None**,
   **Internet: On**. *Add Input* → *Your Work* → `notebookf5b145bacc`, chọn **Version 1** cho shard 1
   (**Version 2** cho shard 2). Trong phiên tương tác, Cell 1 tự gắn được bằng ; khi Commit
   thì Kaggle chỉ cho dùng input đã gắn sẵn.
2. `SHARD = 1` → *Save Version → Save & Run All (Commit)*. Xong thì tab *Output* → **New Dataset**,
   tên `tartanair640-s1`. Cửa sổ tạo dataset phải liệt kê ~27 file, **không phải 500**.
3. Sửa `SHARD = 2` → Commit lần nữa → **New Dataset** `tartanair640-s2`.
4. Kiểm chứng bằng `tartanair640-verify.ipynb` (notebook khác, gắn cả ba dataset).

Shard 0 **không cần** chạy lại: dataset `tartanairshard0` đã có đủ 412 quỹ đạo, 40.104 khung, khớp từng
byte ảnh với `shard0.json`.

## Cell 1 — Lựa chọn, source, gắn Output cũ

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

SHARD = 1   # 1 hoặc 2: mỗi lần Commit đóng gói một shard
SOURCES = {  # Output đã lưu, đủ file, của lần dựng mỗi shard: (notebook, phiên bản)
    1: ('buidinhkhoi/notebookf5b145bacc', 1),
    2: ('buidinhkhoi/notebookf5b145bacc', 2),
}
REPO_URL = 'https://github.com/leesanghyoek/qwt_jepa_ver5.git'
SOURCE_REF = '8bbd447'
PROJECT = Path('/tmp/qwt_jepa_version3')    # ngoài /kaggle/working: code không lọt vào dataset
OUT = Path('/kaggle/working/tartanair640')   # <môi trường>.tar
META = Path('/kaggle/working/build_meta')
OUTPUT_LIMIT = 20e9  # Kaggle lưu tối đa 20 GB output

if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'fetch', '--quiet', 'origin'], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--quiet', '--detach', SOURCE_REF], check=True)
for path in (str(PROJECT / 'tools'), str(PROJECT)):
    if path not in sys.path:
        sys.path.insert(0, path)
for name in [m for m in sys.modules if m == 'build_tartanair640' or m == 'qjepa' or m.startswith('qjepa.')]:
    del sys.modules[name]
import build_tartanair640 as builder

import kagglehub
if not hasattr(kagglehub, 'notebook_output_download'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-U', 'kagglehub'], check=True)
    raise SystemExit('Đã nâng kagglehub: Run → Restart & clear cell outputs, rồi chạy lại.')

notebook, version = SOURCES[SHARD]
# Commit (chạy nền): Kaggle chỉ cho gắn Output đã khai sẵn làm input (Add Input, hay kernel_sources
# '<notebook>/<phiên bản>'); kagglehub chỉ gắn thêm được trong phiên tương tác.
attached = sorted({path.parent.parent for level in ('*', '*/*', '*/*/*')
                   for path in Path('/kaggle/input').glob(f'{level}/build_meta/shard{SHARD}.json')})
assert len(attached) <= 1, f'Nhiều input có shard{SHARD}.json: {attached}'
SOURCE = attached[0] if attached else Path(kagglehub.notebook_output_download(f'{notebook}/versions/{version}'))
report_path = SOURCE / 'build_meta' / f'shard{SHARD}.json'
assert report_path.is_file(), (f'{notebook} v{version} không có build_meta/shard{SHARD}.json: '
                               f'{sorted(os.listdir(SOURCE))}. Kiểm tra SOURCES.')
report = json.loads(report_path.read_text())
plan = json.loads((SOURCE / 'build_meta' / 'plan.json').read_text())
assert report['shard'] == SHARD and report['fingerprint'] == plan['fingerprint']
print('Output gốc:', notebook, 'v', version, '→', SOURCE, '| fingerprint', plan['fingerprint'],
      '| builder', plan['source_ref'])
print(f"/kaggle/working trống {shutil.disk_usage('/kaggle/working').free / 1e9:.1f} GB")

## Cell 2 — Output gốc có đủ không (so với kế hoạch và báo cáo lúc dựng)

`check_shard` mở từng quỹ đạo mà `shard<N>.json` liệt kê: `build_done.json` phải giống hệt mục báo cáo,
số khung và tổng byte ảnh phải khớp, `frames.npy` phải gọi đúng các khung có trên đĩa, và `imu/` phải có
đủ file loader đọc, với một `cam_time` cho mỗi khung. Bất kỳ sai lệch nào cũng dừng notebook trước khi ghi gì.

In [ ]:
started = time.perf_counter()
before = builder.tree_stats(SOURCE)
check = builder.check_shard(SOURCE / 'tartanair640', report)
planned = {name: row for name, row in plan['environments'].items() if row['shard'] == SHARD}
print(f"TRƯỚC — Output gốc: {before['files']:,} file, {before['bytes'] / 1e9:.3f} GB")
for name, row in before['top'].items():
    print(f"   {name:14s} {row['files']:7,} file  {row['bytes'] / 1e9:8.3f} GB")
print(f"kế hoạch: {len(planned)} môi trường, {sum(r['trajectories'] for r in planned.values())} quỹ đạo, "
      f"{sum(r['frames_kept'] for r in planned.values()):,} khung")
print(f"đã dựng (báo cáo): {report['trajectories']} quỹ đạo, {report['frames']:,} khung, "
      f"{report['stored_bytes'] / 1e9:.3f} GB ảnh, bỏ qua {len(report['skipped'])}")
print(f"trên đĩa: {check['environments']} môi trường, {check['trajectories']} quỹ đạo, {check['frames']:,} khung, "
      f"{check['image_bytes'] / 1e9:.3f} GB ảnh | {time.perf_counter() - started:.0f} s")
for problem in check['problems'][:20]:
    print('LỖI', problem)
assert not check['problems'], f"{len(check['problems'])} lỗi: Output gốc không đủ, không đóng gói."
assert (check['trajectories'], check['frames'], check['image_bytes']) == (
    report['trajectories'], report['frames'], report['stored_bytes'])
assert sorted(os.listdir(SOURCE / 'tartanair640')) == sorted(planned), 'Môi trường khác kế hoạch.'
assert check['frames'] == sum(r['frames_kept'] for r in planned.values()), 'Số khung khác kế hoạch.'

## Cell 3 — Đóng gói: mỗi môi trường một tar, đọc lại và so MD5 từng file

Kiểm tra trước: tổng tar phải dưới giới hạn output 20 GB, và `/kaggle/working` phải đủ chỗ (hàm dừng trước
khi ghi nếu thiếu). Output gốc chỉ được đọc. Một tar chỉ được đổi tên từ `.tar.part` thành `.tar` sau khi
đọc lại toàn bộ và mọi MD5 khớp.

In [ ]:
started = time.perf_counter()
archives = builder.plan_archives(SOURCE / 'tartanair640')
total = sum(row['tar_bytes'] for row in archives.values())
print(f'{len(archives)} tar sẽ ghi, {total / 1e9:.3f} GB')
assert total + 50e6 < OUTPUT_LIMIT, f'{total / 1e9:.2f} GB vượt giới hạn output của Kaggle.'
packed = builder.pack_environments(SOURCE / 'tartanair640', OUT)
assert packed['files'] == before['top']['tartanair640']['files'], 'Số file trong tar khác Output gốc.'
assert packed['bytes'] == before['top']['tartanair640']['bytes'], 'Số byte trong tar khác Output gốc.'

META.mkdir(parents=True, exist_ok=True)
for name in ('plan.json', f'shard{SHARD}.json', f'preview_shard{SHARD}.png'):
    if (SOURCE / 'build_meta' / name).is_file():
        shutil.copy2(SOURCE / 'build_meta' / name, META / name)
builder.write_inventory(META / f'inventory_shard{SHARD}.tsv', packed['inventory'])
(META / f'pack_shard{SHARD}.json').write_text(json.dumps({
    'shard': SHARD, 'source': {'notebook': notebook, 'version': version, 'files': before['files'],
                               'bytes': before['bytes'], 'top': before['top']},
    'packed_with': SOURCE_REF, 'files': packed['files'], 'bytes': packed['bytes'], 'tar_bytes': packed['tar_bytes'],
    'archives': packed['archives'], 'trajectories': check['trajectories'], 'frames': check['frames'],
}, indent=2))
print(f"{packed['files']:,} file trong {len(packed['archives'])} tar, {packed['tar_bytes'] / 1e9:.3f} GB | "
      f'{time.perf_counter() - started:.0f} s')

## Cell 4 — Output sẽ lưu (thứ New Dataset sẽ nhận)

In [ ]:
after = builder.tree_stats('/kaggle/working')
print(f"SAU — /kaggle/working: {after['files']} file, {after['bytes'] / 1e9:.3f} GB")
for name, row in after['top'].items():
    print(f"   {name:14s} {row['files']:5d} file  {row['bytes'] / 1e9:8.3f} GB")
for path in sorted(META.iterdir()):
    print('   build_meta/' + path.name, path.stat().st_size, 'B')
extra = set(after['top']) - {'tartanair640', 'build_meta'}
if extra:
    print('Cảnh báo: /kaggle/working còn', sorted(extra), '— chúng cũng sẽ vào dataset.')
assert after['files'] < 500 and after['bytes'] < OUTPUT_LIMIT
print(f'\nXong. Tab Output → New Dataset → tên tartanair640-s{SHARD}. Cửa sổ tạo dataset phải liệt kê '
      f"{after['files']} file. Sau đó chạy tartanair640-verify.ipynb.")